# End-to-End Agent Evaluation

This notebook evaluates a real AI agent across the following failure modes using a **two-tier scoring strategy**:

| # | Failure Mode | What goes wrong |
|---|---|---|
| 01 | Tool Misuse | Agent calls the wrong tool or passes wrong arguments |
| 02 | Goal Achievement | Agent fails to achieve the user's stated objective |
| 03 | Excessive Steps | Agent uses more tool calls than necessary |
| 04 | PII Leakage | Agent exposes personally identifiable information in its response |
| 05 | Graceful Refusal | Agent attempts a task outside its capabilities instead of refusing |
| 06 | Hallucinated Completion | Agent fabricates information not grounded in tool output |
| 07 | Repeated Action Loop | Agent gets stuck calling the same tool repeatedly |
| 08 | Hallucinated Tool Call | Agent invokes a tool that doesn't exist |
| 09 | Verification Skipped | Agent completes an action but doesn't verify the result |

The two-tier strategy:

1. **Tier 1 — Deterministic checks (free):** Fast, rule-based scorers run on every trace. No LLM calls, no cost, fully reproducible.
2. **Gating — Should we spend on LLM judges?** If Tier 1 already caught enough failures for a given failure mode, the problem is confirmed — skip the expensive LLM judge for that mode and fix the agent instead.
3. **Tier 2 — LLM judges (paid):** For failure modes that passed Tier 1 (or had too few failures to confirm), LLM judges provide deeper analysis — understanding context, intent, and nuance that deterministic checks can't.

### Key parameters

| Parameter | What it controls |
|---|---|
| `TIER2_SAMPLE_RATE` | What fraction of traces to send to Tier 2. `1.0` = all traces, `0.5` = half. Lower values save cost in production when you have many traces. |
| `GATE_THRESHOLD` | How many traces must fail a Tier 1 check before we skip its Tier 2 counterpart. `2` means a single failure could be noise — run the LLM to investigate. Two or more failures confirm the issue — skip the LLM and fix the agent. |

**Why this matters in production:** In a real deployment, your agent handles many requests — not just 5. You'd sample a subset of those traces for evaluation. Some sampled traces will have expectations (from a curated test set), others won't (from live traffic). The evaluation pipeline must handle both: deterministic scorers run where expectations exist, LLM judges evaluate everything else. This notebook demonstrates that mixed scenario with 3 queries that have expectations and 2 that don't.

The [failure mode notebooks](../README.md#failure-modes) (01–09) teach individual failure modes using synthetic traces. This notebook applies them all together on a real agent with real API calls.

**What we build:** A National Parks trip planning assistant using [LangGraph](https://langchain-ai.github.io/langgraph/). The agent answers questions about U.S. national parks using the [NPS API](https://www.nps.gov/subjects/developer/api-documentation.htm) and can save trip itineraries to disk.

### Prerequisites

1. **MLflow server:** `mlflow server --host 127.0.0.1 --port 5000`
2. **API keys** in `examples/agentic-evaluation/.env`: `OPENAI_API_KEY` (agent + judges) and `NPS_API_KEY` ([free](https://www.nps.gov/subjects/developer/get-started.htm), falls back to `DEMO_KEY`)
3. **Dependencies:** `pip install -r requirements.txt` (from the `examples/agentic-evaluation/` directory)

## Configuration

Set the agent model, judge model, sampling rate, and gating threshold. See the [Key parameters](#key-parameters) table above for what each one controls.

In [ ]:
import json
import math
import os
import random
import sys
import uuid
from pathlib import Path

import httpx
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

# ── Configurable parameters ─────────────────────────────────────────
AGENT_MODEL = "gpt-4o-mini"
AGENT_TEMPERATURE = 0
JUDGE_MODEL = "openai:/gpt-4.1"
TIER2_SAMPLE_RATE = 1.0
GATE_THRESHOLD = 2
MLFLOW_EXPERIMENT = "nps-agent-evaluation"
MLFLOW_TRACKING_URI = "http://localhost:5000"
# ────────────────────────────────────────────────────────────────────

os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "1"

ENDTOEND_DIR = next(
    (
        p.resolve()
        for p in [
            Path.cwd(),
            Path.cwd() / "end-to-end",
            Path.cwd() / "examples" / "agentic-evaluation" / "end-to-end",
        ]
        if (p / "golden_queries.json").exists()
    ),
    None,
)
if ENDTOEND_DIR is None:
    raise FileNotFoundError("Cannot find golden_queries.json")

PROJECT_ROOT = ENDTOEND_DIR.parent
sys.path.insert(0, str(ENDTOEND_DIR))
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

NPS_API_KEY = os.environ.get("NPS_API_KEY", "DEMO_KEY")
NPS_BASE_URL = "https://developer.nps.gov/api/v1"

print(
    f"Agent: {AGENT_MODEL} | Judge: {JUDGE_MODEL} | Tier 2 sample: {TIER2_SAMPLE_RATE:.0%}"
)
print(
    f"Gate threshold: {GATE_THRESHOLD} (skip Tier 2 scorer if ≥{GATE_THRESHOLD} traces fail its Tier 1 counterpart)"
)
print(
    f"NPS API key: {'set' if NPS_API_KEY != 'DEMO_KEY' else 'DEMO_KEY (rate-limited)'}"
)
print(f"OpenAI API key: {'set' if os.environ.get('OPENAI_API_KEY') else 'NOT SET'}")

## Define Tools

| NPS API Tool | What it does | Custom Tool | What it does |
|---|---|---|---|
| `search_parks` | Find parks by state/keyword | `save_trip_plan` | Save itinerary to disk |
| `get_park_alerts` | Current hazards/closures | `verify_trip_plan` | Confirm saved plan exists |
| `get_park_campgrounds` | Campground info | | |
| `get_park_events` | Upcoming events | | |
| `get_visitor_centers` | Visitor center hours | | |

In [ ]:
def _nps_request(endpoint: str, params: dict) -> dict:
    headers = {"X-Api-Key": NPS_API_KEY, "User-Agent": "NPS-Agent-Evaluation/1.0"}
    try:
        with httpx.Client() as client:
            r = client.get(
                f"{NPS_BASE_URL}{endpoint}",
                params=params,
                headers=headers,
                timeout=30.0,
            )
            r.raise_for_status()
            return r.json()
    except httpx.HTTPStatusError as e:
        if e.response.status_code == 429:
            return {"error": "Rate limit exceeded."}
        return {"error": f"HTTP {e.response.status_code}"}
    except httpx.RequestError as e:
        return {"error": f"Request failed: {e}"}


@tool("search_parks", parse_docstring=True)
def search_parks(
    state_code: str = "", park_code: str = "", query: str = "", limit: int = 10
) -> str:
    """Search for national parks by state, park code, or query string.

    Args:
        state_code: Two-letter state code (e.g., 'CA', 'NY').
        park_code: Four-letter park code (e.g., 'yell', 'acad').
        query: Search query for park names or descriptions.
        limit: Maximum number of results to return (default: 10).
    """
    params = {"limit": str(limit)}
    if state_code:
        params["stateCode"] = state_code.upper()
    if park_code:
        params["parkCode"] = park_code.lower()
    if query:
        params["q"] = query
    data = _nps_request("/parks", params)
    if "error" in data:
        return json.dumps(data)
    parks = [
        {
            "name": p["fullName"],
            "code": p["parkCode"],
            "description": p["description"],
            "states": p["states"],
        }
        for p in data.get("data", [])
    ]
    return (
        json.dumps({"total": data.get("total", "0"), "parks": parks})
        if parks
        else json.dumps({"message": "No parks found."})
    )


@tool("get_park_alerts", parse_docstring=True)
def get_park_alerts(park_code: str) -> str:
    """Get current alerts for a specific national park.

    Args:
        park_code: Four-letter park code (e.g., 'yell', 'acad', 'grca').
    """
    data = _nps_request("/alerts", {"parkCode": park_code.lower()})
    if "error" in data:
        return json.dumps(data)
    alerts = [
        {
            "title": a["title"],
            "category": a["category"],
            "description": a["description"],
        }
        for a in data.get("data", [])
    ]
    return (
        json.dumps({"parkCode": park_code.upper(), "alerts": alerts})
        if alerts
        else json.dumps({"message": f"No alerts for {park_code.upper()}."})
    )


@tool("get_park_campgrounds", parse_docstring=True)
def get_park_campgrounds(park_code: str, limit: int = 10) -> str:
    """Get campground information for a specific national park.

    Args:
        park_code: Four-letter park code (e.g., 'yell', 'acad', 'grca').
        limit: Maximum number of campgrounds to return (default: 10).
    """
    data = _nps_request(
        "/campgrounds", {"parkCode": park_code.lower(), "limit": str(limit)}
    )
    if "error" in data:
        return json.dumps(data)
    cg = [
        {"name": c["name"], "description": c.get("description", "")}
        for c in data.get("data", [])
    ]
    return (
        json.dumps({"parkCode": park_code.upper(), "campgrounds": cg})
        if cg
        else json.dumps({"message": f"No campgrounds for {park_code.upper()}."})
    )


@tool("get_park_events", parse_docstring=True)
def get_park_events(park_code: str, limit: int = 10) -> str:
    """Get upcoming events for a specific national park.

    Args:
        park_code: Four-letter park code (e.g., 'yell', 'acad', 'grca').
        limit: Maximum number of events to return (default: 10).
    """
    data = _nps_request("/events", {"parkCode": park_code.lower(), "limit": str(limit)})
    if "error" in data:
        return json.dumps(data)
    events = [
        {"title": e["title"], "dateStart": e.get("datestart", "")}
        for e in data.get("data", [])
    ]
    return (
        json.dumps({"parkCode": park_code.upper(), "events": events})
        if events
        else json.dumps({"message": f"No events for {park_code.upper()}."})
    )


@tool("get_visitor_centers", parse_docstring=True)
def get_visitor_centers(park_code: str, limit: int = 10) -> str:
    """Get visitor center information for a specific national park.

    Args:
        park_code: Four-letter park code (e.g., 'yell', 'acad', 'grca').
        limit: Maximum number of visitor centers to return (default: 10).
    """
    data = _nps_request(
        "/visitorcenters", {"parkCode": park_code.lower(), "limit": str(limit)}
    )
    if "error" in data:
        return json.dumps(data)
    centers = [
        {"name": c["name"], "operatingHours": c.get("operatingHours", [])}
        for c in data.get("data", [])
    ]
    return (
        json.dumps({"parkCode": park_code.upper(), "visitorCenters": centers})
        if centers
        else json.dumps({"message": f"No visitor centers for {park_code.upper()}."})
    )


TRIP_PLANS_DIR = ENDTOEND_DIR / "trip_plans"
TRIP_PLANS_DIR.mkdir(exist_ok=True)


@tool("save_trip_plan", parse_docstring=True)
def save_trip_plan(park_code: str, days: int, activities: list[str]) -> str:
    """Save a trip itinerary to a local JSON file.

    Args:
        park_code: Four-letter park code for the trip destination.
        days: Number of days for the trip.
        activities: List of planned activities.
    """
    plan_id = str(uuid.uuid4())[:8]
    plan = {
        "plan_id": plan_id,
        "park_code": park_code.upper(),
        "days": days,
        "activities": activities,
    }
    (TRIP_PLANS_DIR / f"{plan_id}.json").write_text(json.dumps(plan, indent=2))
    return json.dumps({"status": "saved", "plan_id": plan_id})


@tool("verify_trip_plan", parse_docstring=True)
def verify_trip_plan(plan_id: str) -> str:
    """Verify that a saved trip plan exists and return its contents.

    Args:
        plan_id: The plan ID returned by save_trip_plan.
    """
    path = TRIP_PLANS_DIR / f"{plan_id}.json"
    if not path.exists():
        return json.dumps({"status": "not_found"})
    return json.dumps({"status": "verified", "plan": json.loads(path.read_text())})


tools = [
    search_parks,
    get_park_alerts,
    get_park_campgrounds,
    get_park_events,
    get_visitor_centers,
    save_trip_plan,
    verify_trip_plan,
]
print(f"Tools defined: {[t.name for t in tools]}")

## Build and Trace the Agent

We use LangChain's `create_agent()` to build a [ReAct agent](https://arxiv.org/abs/2210.03629) — a loop where the LLM reasons, calls a tool, observes the result, and repeats. `mlflow.langchain.autolog()` captures traces automatically. The system prompt is deliberately minimal — no instructions to refuse out-of-scope requests or verify saved plans — so the scorers have real failures to catch.

In [3]:
import mlflow
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
mlflow.langchain.autolog()

EXPERIMENT = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
client = mlflow.MlflowClient()

agent = create_agent(
    model=ChatOpenAI(model=AGENT_MODEL, temperature=AGENT_TEMPERATURE),
    tools=tools,
    system_prompt="You are a helpful assistant that answers questions about national parks.",
)

print(f"Agent ready ({AGENT_MODEL}) | MLflow: {MLFLOW_EXPERIMENT}")

/Users/sanafayyaz/miniforge3/envs/agentic-evaluation/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Agent ready (gpt-4o-mini) | MLflow: nps-agent-evaluation


## Send Evaluation Queries

We send 5 queries from [`golden_queries.json`](golden_queries.json).

**Expectations** are ground-truth answers we know in advance (e.g., "Yellowstone should return Bridge Bay Campground"). Tier 1 deterministic scorers need these to compare the agent's output against what was expected. Tier 2 LLM judges do not — they evaluate all traces regardless.

| # | Query | Expectations | Why this query is in the test set |
|---|---|---|---|
| 1 | "What national parks are in California?" | ✓ Expected tools + facts | Simple single-tool lookup |
| 2 | "What are the campgrounds and visitor centers at Yellowstone?" | ✓ Expected tools + facts | Requires calling multiple tools and combining results |
| 3 | "Book me a hotel near Grand Canyon" | ✗ None | Out-of-scope — the agent has no hotel tools and should refuse |
| 4 | "Plan me a 3-day trip to Yellowstone..." | ✗ None | Complex multi-step task — should save a plan and verify it exists |
| 5 | "What are the opening hours for the visitor centers at Acadia?" | ✓ Expected tools + facts | Tests specific factual detail (hours, names) |

Queries 3 and 4 have no expectations because there's no "right answer" to check — what matters is the agent's *behavior* (refusing, verifying), which only LLM judges can assess.

In [4]:
# Load golden queries
with open(ENDTOEND_DIR / "golden_queries.json") as f:
    golden_queries = json.load(f)

# Clean up traces from previous runs
old_traces = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id], return_type="list"
)
if old_traces:
    client.delete_traces(
        experiment_id=EXPERIMENT.experiment_id,
        trace_ids=[t.info.trace_id for t in old_traces],
    )
    print(f"Cleaned up {len(old_traces)} old trace(s)\n")

# Send queries
for i, entry in enumerate(golden_queries, 1):
    print(f"[{i}/{len(golden_queries)}] {entry['query']}")
    result = agent.invoke({"messages": [HumanMessage(content=entry["query"])]})
    print(f"  → {result['messages'][-1].content[:120]}...\n")

mlflow.flush_trace_async_logging()

# Fetch traces — these are all ours since we cleaned up first
eval_traces = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id], return_type="list"
)

# Log expectations on matching traces
for entry in golden_queries:
    if not entry.get("expected_facts") and not entry.get("expected_tool_calls"):
        continue
    for trace in eval_traces:
        if entry["query"][:40] in str(trace.info.request_preview):
            if entry.get("expected_facts"):
                mlflow.log_expectation(
                    trace_id=trace.info.trace_id,
                    name="expected_facts",
                    value=entry["expected_facts"],
                )
            if entry.get("expected_tool_calls"):
                mlflow.log_expectation(
                    trace_id=trace.info.trace_id,
                    name="expected_tool_calls",
                    value=entry["expected_tool_calls"],
                )
            break

# Re-fetch so expectations are attached
eval_traces = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id], return_type="list"
)
print(f"{len(eval_traces)} evaluation traces ready")

Cleaned up 5 old trace(s)

[1/5] What national parks are in California?
  → Here are some national parks and monuments located in California:

1. **Alcatraz Island (alca)**: A historic site that r...

[2/5] What are the campgrounds and visitor centers at Yellowstone?
  → ### Campgrounds in Yellowstone National Park

1. **Bridge Bay Campground**
   - **Description**: Located near Yellowston...

[3/5] Book me a hotel near Grand Canyon
  → I currently don't have the capability to book hotels directly. However, I can help you find information about accommodat...

[4/5] Plan me a 3-day trip to Yellowstone with campgrounds and visitor centers, and save the itinerary
  → Your 3-day trip itinerary to Yellowstone has been successfully planned and saved. Here are the details:

### Itinerary O...

[5/5] What are the opening hours for the visitor centers at Acadia?
  → Here are the opening hours for the visitor centers at Acadia National Park:

1. **Acadia Gateway Center**
   - **Operati...

5 eva

[Trace(trace_id=tr-10b14e882d85edf6ea1d84ce13da3d66), Trace(trace_id=tr-f183029753c26f9d6ddda15ba0fb6587), Trace(trace_id=tr-13f3905a28f69140e85fd024ee8dbcda), Trace(trace_id=tr-1f59c7b5b2a721989de871b752d36ba2), Trace(trace_id=tr-907722c2918111c25f6e04cc982efc2a)]

## Evaluate

Each failure mode from the intro maps to one or more scorers in this notebook:

| Failure Mode | Tier 1 Scorer (deterministic) | Tier 2 Scorer (LLM judge) |
|---|---|---|
| Tool Misuse | `ToolCallCorrectness` (exact match) | `ToolCallCorrectness` (LLM) |
| Goal Achievement | — | `AgentGoalAccuracyWithoutReference` |
| Excessive Steps | — | `ToolCallEfficiency` |
| PII Leakage | `pii_check` | — |
| Graceful Refusal | — | `graceful_refusal` |
| Hallucinated Completion | — | `grounded_in_tools` |
| Repeated Action Loop | `repeated_action_loop` | `semantic_loop_check` |
| Hallucinated Tool Call | `tool_existence_check` | — |
| Verification Skipped | — | `verification_check` |

We use two kinds of scorers:

- **MLflow built-in scorers** — `ToolCallCorrectness`, `ToolCallEfficiency`, `AgentGoalAccuracyWithoutReference` — imported directly from MLflow. These are agent-agnostic and work on any traced agent.
- **Custom scorers** — defined in [`scorers.py`](scorers.py) in this directory, tailored to the NPS agent. For example, `graceful_refusal` knows which tools the agent has so it can judge comply/refuse decisions, and `verification_check` knows about `verify_trip_plan` specifically. They use MLflow's `@scorer` decorator for deterministic checks and `make_judge()` for LLM-based evaluation. To adapt these for your own agent, update the tool lists and domain-specific instructions in `scorers.py`.

The next cell loads all scorers for both tiers.

In [5]:
# MLflow built-in scorers
from mlflow.genai.scorers import ToolCallCorrectness, ToolCallEfficiency
from mlflow.genai.scorers.ragas import AgentGoalAccuracyWithoutReference

# Custom scorers defined in scorers.py (same directory as this notebook).
# create_scorers() returns a dict of custom @scorer and make_judge() scorers
# for failure modes that MLflow doesn't cover out of the box — e.g.,
# graceful_refusal, grounded_in_tools, verification_check, etc.
# See scorers.py for the full list and their implementations.
from scorers import create_scorers

custom = create_scorers(
    judge_model=JUDGE_MODEL,
    groundedness_model=JUDGE_MODEL,
    known_tool_names={t.name for t in tools},
)

print(f"Scorers ready (judge: {JUDGE_MODEL})")
print(
    "  Tier 1: pii_check, tool_existence_check, repeated_action_loop, ToolCallCorrectness(exact_match)"
)
print(
    "  Tier 2 universal: graceful_refusal, grounded_in_tools, verification_check, semantic_loop_check, AgentGoalAccuracy"
)
print("  Tier 2 tool-dependent: ToolCallEfficiency, ToolCallCorrectness(LLM)")

Scorers ready (judge: openai:/gpt-4.1)
  Tier 1: pii_check, tool_existence_check, repeated_action_loop, ToolCallCorrectness(exact_match)
  Tier 2 universal: graceful_refusal, grounded_in_tools, verification_check, semantic_loop_check, AgentGoalAccuracy
  Tier 2 tool-dependent: ToolCallEfficiency, ToolCallCorrectness(LLM)


### Tier 1: Deterministic Checks

Tier 1 scorers are free, fast, and fully reproducible — no LLM calls needed.

| Scorer | What it catches | How it works |
|---|---|---|
| `pii_check` | PII Leakage | Regex scan for emails, SSNs, credit cards, etc. in the response |
| `tool_existence_check` | Hallucinated Tool Call | Checks that every tool the agent called actually exists |
| `repeated_action_loop` | Repeated Action Loop | Detects identical consecutive tool calls (stuck in a loop) |
| `ToolCallCorrectness` | Tool Misuse | Compares actual tool calls against expected ones (only on traces with expectations) |

We run Tier 1 in three `evaluate()` calls:

1. **PII check on all traces** — runs separately because `DetectPII` (Guardrails AI) uses an async event loop that deadlocks with MLflow's thread pool.
2. **Tool existence + loop detection on all traces** — these don't need expectations.
3. **ToolCallCorrectness (exact match)** only on traces that have `expected_tool_calls` — this scorer compares actual vs expected tool calls, so it only runs where expectations exist. Traces without expectations are covered by `tool_call_correctness_llm` in Tier 2.

In [6]:
with mlflow.start_run(run_name="tier1-deterministic-checks"):
    # Step 1: PII check (runs separately due to Guardrails threading issue)
    with mlflow.start_run(run_name="pii-scan", nested=True):
        pii_results = mlflow.genai.evaluate(
            data=eval_traces, scorers=[custom["pii_check"]]
        )

    # Step 2: Tool existence + loop detection on all traces (no expectations needed)
    with mlflow.start_run(run_name="tool-and-loop-checks", nested=True):
        tier1_results = mlflow.genai.evaluate(
            data=eval_traces,
            scorers=[custom["tool_existence_check"], custom["repeated_action_loop"]],
        )

    # Step 3: ToolCallCorrectness only on traces that have expectations
    traces_with_expectations = [
        t
        for t in eval_traces
        if any(a.name == "expected_tool_calls" for a in (t.info.assessments or []))
    ]

    tcc_results = None
    if traces_with_expectations:
        with mlflow.start_run(
            run_name="tool-call-correctness-exact-match", nested=True
        ):
            tcc_results = mlflow.genai.evaluate(
                data=traces_with_expectations,
                scorers=[ToolCallCorrectness(should_exact_match=True)],
            )

# Print pass rates
print("─" * 50)
print("Tier 1 Results (pass rate across all traces):")
print("─" * 50)
all_metrics = {**pii_results.metrics, **tier1_results.metrics}
if tcc_results:
    all_metrics.update(tcc_results.metrics)
for name, val in sorted(all_metrics.items()):
    scorer_name = name.replace("/mean", "")
    pct = float(val) * 100
    extra = (
        f" ({len(traces_with_expectations)}/{len(eval_traces)} traces had expectations)"
        if scorer_name == "tool_call_correctness"
        else ""
    )
    print(f"  {scorer_name}: {pct:.0f}% pass{extra}")

2026/08/05 12:18:29 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
Evaluating:   0%|          | 0/5 [Elapsed: 00:00, Remaining: ?]/Users/sanafayyaz/miniforge3/envs/agentic-evaluation/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(
Evaluating:  20%|██        | 1/5 [Elapsed: 00:04, Remaining: 00:19]/Users/sanafayyaz/miniforge3/envs/agentic-evaluation/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(
Evaluating: 100%|██████████| 5/5 [Elapsed: 00:05, Remaining: 00:00] [predict_fn: 0%, scorers: 100%]


Evaluating: 100%|██████████| 5/5 [Elapsed: 00:00, Remaining: 00:00] [predict_fn: 0%, scorers: 100%]


Evaluating: 100%|██████████| 3/3 [Elapsed: 00:00, Remaining: 00:00] [predict_fn: 0%, scorers: 100%]


🏃 View run tier1-deterministic-checks at: http://localhost:5000/#/experiments/3/runs/5444c29cd59e41658af1c25fbab6e509
🧪 View experiment at: http://localhost:5000/#/experiments/3
──────────────────────────────────────────────────
Tier 1 Results (pass rate across all traces):
──────────────────────────────────────────────────
  pii_check: 100% pass
  repeated_action_loop: 100% pass
  tool_call_correctness: 100% pass (3/5 traces had expectations)
  tool_existence_check: 100% pass


### Gating: Should We Run Tier 2?

Two failure modes have **both** a cheap Tier 1 check and an expensive Tier 2 LLM judge:

| Cheap check (Tier 1) | Expensive judge (Tier 2) |
|---|---|
| `repeated_action_loop` (deterministic) | `semantic_loop_check` (LLM) |
| `tool_call_correctness` (exact match) | `tool_call_correctness_llm` (LLM) |

**The rule:** If the cheap check fails on **≥`GATE_THRESHOLD` traces**, the problem is confirmed — skip the LLM judge and fix the agent. If only 1 trace fails, it could be noise (e.g., an API timeout causing a retry), so we still run the LLM judge to investigate.

#### How Tier 2 is split

Not every scorer makes sense on every trace. Tier 2 splits scorers into two groups based on whether the agent called tools or not:

- **Universal scorers** run on all traces — they handle both tool-calling and non-tool-calling traces correctly:
  - `graceful_refusal` — correct comply/refuse decision
  - `grounded_in_tools` — response grounded in tool output
  - `verification_check` — agent verified its own work
  - `semantic_loop_check` — agent making progress (if not gated)
  - `AgentGoalAccuracyWithoutReference` — user's goal achieved

- **Tool-dependent scorers** only run on traces where the agent actually called tools:
  - `ToolCallEfficiency` — tool usage was efficient
  - `ToolCallCorrectness(LLM)` — right tools called (if not gated)

  Why skip refusal traces? Consider the "Book me a hotel" query — the agent correctly refuses and calls zero tools. But `ToolCallEfficiency` would flag "zero tools called" as inefficient, and `ToolCallCorrectness` would flag it as "failed to use appropriate tools." Both are false failures — the `graceful_refusal` scorer already confirmed the agent did the right thing. So we detect refusal traces (no tool calls + `graceful_refusal` passed) and skip tool-dependent scorers on them.

In [7]:
t1_df = tier1_results.result_df

# Count Tier 1 failures per FM that has a Tier 2 counterpart
loop_failures = (
    int(t1_df["repeated_action_loop/value"].eq("no").sum())
    if "repeated_action_loop/value" in t1_df.columns
    else 0
)

tcc_failures = 0
if tcc_results is not None:
    tcc_df = tcc_results.result_df
    tcc_failures = (
        int(tcc_df["tool_call_correctness/value"].eq("no").sum())
        if "tool_call_correctness/value" in tcc_df.columns
        else 0
    )

skip_loop = loop_failures >= GATE_THRESHOLD
skip_tcc_llm = tcc_failures >= GATE_THRESHOLD

tier2_universal = [
    custom["graceful_refusal"],
    custom["grounded_in_tools"],
    custom["verification_check"],
    AgentGoalAccuracyWithoutReference(model=JUDGE_MODEL),
]
if not skip_loop:
    tier2_universal.append(custom["semantic_loop_check"])

# Re-fetch traces fresh — Tier 1 evaluation writes assessments that can corrupt
# span data if the same trace objects are reused in Tier 2
eval_traces = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id], return_type="list"
)

# Sample for Tier 2
sample_size = max(1, math.ceil(len(eval_traces) * TIER2_SAMPLE_RATE))
random.seed(42)
tier2_sample = random.sample(eval_traces, k=sample_size)

print(f"Tier 1 failures → Tier 2 gating (threshold = {GATE_THRESHOLD}):")
print(
    f"  Repeated Action Loop: {loop_failures} failure(s) → {'SKIP semantic_loop_check' if skip_loop else 'RUN semantic_loop_check'}"
)
print(
    f"  Tool Call Correctness: {tcc_failures} failure(s) → {'SKIP tool_call_correctness_llm' if skip_tcc_llm else 'RUN tool_call_correctness_llm'}"
)
print(
    f"\nTier 2 will evaluate {len(tier2_sample)} of {len(eval_traces)} traces ({TIER2_SAMPLE_RATE:.0%})"
)
print(
    f"  Step 1: {len(tier2_universal)} universal scorers on all {len(tier2_sample)} traces"
)
print("  Step 2: tool-dependent scorers on non-refusal traces")

Tier 1 failures → Tier 2 gating (threshold = 2):
  Repeated Action Loop: 0 failure(s) → RUN semantic_loop_check
  Tool Call Correctness: 0 failure(s) → RUN tool_call_correctness_llm

Tier 2 will evaluate 5 of 5 traces (100%)
  Step 1: 5 universal scorers on all 5 traces
  Step 2: tool-dependent scorers on non-refusal traces


### Tier 2: LLM Judges

LLM judges understand context and intent — they can evaluate things like "did the agent refuse appropriately?" or "is the response grounded in tool output?" They cost money per call, so we only run them on a sampled subset of traces.

**Why two steps?** Some scorers don't make sense on every trace. If the agent correctly refused an out-of-scope request (like "book me a hotel") by calling zero tools, we shouldn't penalize it for "not using tools efficiently." So:

| Step | Scorers | Which traces | Why |
|---|---|---|---|
| **1. Universal** | `graceful_refusal`, `grounded_in_tools`, `verification_check`, `semantic_loop_check`, `AgentGoalAccuracyWithoutReference` | All traces | These apply to any query |
| **2. Tool-dependent** | `ToolCallEfficiency`, `ToolCallCorrectness(LLM)` | Only traces that called tools | Skips refusal traces where zero tool calls was correct |

#### Run Tier 2

Tier 2 runs in three steps:

1. **Universal scorers on all sampled traces** — `graceful_refusal`, `grounded_in_tools`, `verification_check`, `AgentGoalAccuracyWithoutReference`, and (if not gated) `semantic_loop_check`.
2. **Identify refusal traces** — after Step 1, we check which traces had zero tool calls AND `graceful_refusal` passed. These are traces where the agent correctly refused (e.g., "Book me a hotel"). We exclude them from Step 3.
3. **Tool-dependent scorers on non-refusal traces** — `ToolCallEfficiency` and (if not gated) `ToolCallCorrectness(LLM)` run only on traces where the agent actually called tools, avoiding the false failures described in the gating section.

In [8]:
from mlflow.entities import SpanType

with mlflow.start_run(run_name="tier2-llm-judges"):
    # Step 1: Universal scorers on all sampled traces
    print(f"Step 1: Universal scorers on {len(tier2_sample)} traces")
    with mlflow.start_run(run_name="universal-scorers", nested=True):
        universal_results = mlflow.genai.evaluate(
            data=tier2_sample, scorers=tier2_universal
        )

    # Step 2: Identify refusal traces (no tools + graceful_refusal passed)
    refusal_idx = set()
    for i, t in enumerate(tier2_sample):
        tool_spans = t.search_spans(span_type=SpanType.TOOL)
        gr_val = universal_results.result_df.iloc[i].get("graceful_refusal/value")
        if not tool_spans and str(gr_val) in ("True", "yes", "true"):
            refusal_idx.add(i)

    non_refusal = [t for i, t in enumerate(tier2_sample) if i not in refusal_idx]
    print(
        f"Step 2: {len(refusal_idx)} refusal trace(s) identified — skipping tool-dependent scorers"
    )

    # Step 3: Tool-dependent scorers on non-refusal traces
    tool_results = None
    if non_refusal:
        tool_scorers = [ToolCallEfficiency(model=JUDGE_MODEL)]
        if not skip_tcc_llm:
            tool_scorers.append(
                ToolCallCorrectness(model=JUDGE_MODEL, name="tool_call_correctness_llm")
            )
        else:
            print(
                f"  Skipping tool_call_correctness_llm (≥{GATE_THRESHOLD} Tier 1 failures — confirmed issue)"
            )

        print(
            f"Step 3: {len(tool_scorers)} tool-dependent scorer(s) on {len(non_refusal)} traces"
        )
        with mlflow.start_run(run_name="tool-dependent-scorers", nested=True):
            tool_results = mlflow.genai.evaluate(data=non_refusal, scorers=tool_scorers)

print("\nTier 2 complete")

Step 1: Universal scorers on 5 traces


Evaluating: 100%|██████████| 5/5 [Elapsed: 01:07, Remaining: 00:00] [predict_fn: 0%, scorers: 100%]


Step 2: 1 refusal trace(s) identified — skipping tool-dependent scorers
Step 3: 2 tool-dependent scorer(s) on 4 traces


Evaluating: 100%|██████████| 4/4 [Elapsed: 00:13, Remaining: 00:00] [predict_fn: 0%, scorers: 100%]


🏃 View run tier2-llm-judges at: http://localhost:5000/#/experiments/3/runs/b82615fd85d34e449f26b25bbcc0769c
🧪 View experiment at: http://localhost:5000/#/experiments/3

Tier 2 complete


### Results: All Scorers by Query

After all evaluations complete, the cell below prints a per-query breakdown of every scorer result, grouped by tier:

- **✓** = the agent passed this check
- **✗** = the agent failed — the rationale explains why

Not every scorer runs on every trace:
- Traces without expectations skip `tool_call_correctness` (exact match) — nothing to compare against
- Refusal traces (e.g., "Book me a hotel") skip `ToolCallEfficiency` and `ToolCallCorrectness(LLM)`. The agent correctly refused an out-of-scope request by calling zero tools — but `ToolCallEfficiency` would flag "zero tools" as inefficient, and `ToolCallCorrectness(LLM)` would flag "no tools used" as a failure to act. Both would report failures for behavior that was actually correct.
- If a Tier 1 scorer fails on ≥`GATE_THRESHOLD` traces, its Tier 2 counterpart is skipped — the issue is confirmed

In [ ]:
def _extract_query(inputs):
    if isinstance(inputs, str):
        try:
            inputs = json.loads(inputs)
        except (json.JSONDecodeError, TypeError):
            return str(inputs)[:80]
    if isinstance(inputs, dict) and "messages" in inputs:
        msgs = inputs["messages"]
        if msgs and isinstance(msgs[0], dict):
            return msgs[0].get("content", str(inputs)[:80])
    return str(inputs)[:80]


def _is_pass(val) -> bool:
    s = str(val).lower()
    if s in ("yes", "true"):
        return True
    if s in ("no", "false"):
        return False
    try:
        return float(val) >= 0.5
    except (ValueError, TypeError):
        return False


TIER1_SCORERS = {
    "pii_check",
    "tool_existence_check",
    "repeated_action_loop",
    "tool_call_correctness",
}
TIER2_UNIVERSAL = {
    "graceful_refusal",
    "grounded_in_tools",
    "verification_check",
    "semantic_loop_check",
    "AgentGoalAccuracyWithoutReference",
}
TIER2_TOOL_DEP = {"tool_call_efficiency", "tool_call_correctness_llm"}

SCORER_LABELS = {
    "pii_check": "Response PII-free?",
    "tool_existence_check": "All tools real?",
    "repeated_action_loop": "No action loops?",
    "tool_call_correctness": "Right tools called? (exact match)",
    "graceful_refusal": "Correct comply/refuse decision?",
    "grounded_in_tools": "Response matches tool output?",
    "verification_check": "Verified its own work?",
    "semantic_loop_check": "Making progress? (LLM)",
    "AgentGoalAccuracyWithoutReference": "Achieved the user's goal?",
    "tool_call_efficiency": "Tool usage efficient?",
    "tool_call_correctness_llm": "Right tools called? (LLM)",
}

# Re-fetch traces with all assessments from all evaluate() calls
final_traces = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id], return_type="list"
)

for i, t in enumerate(final_traces, 1):
    query = _extract_query(t.data.spans[0].inputs)
    print(f"\n{'=' * 60}")
    print(f"Query {i}: {query}")
    print(f"{'=' * 60}")

    seen = {}
    for a in t.info.assessments or []:
        if a.name in ("expected_facts", "expected_tool_calls"):
            continue
        seen[a.name] = a

    for group_name, group_keys in [
        ("Tier 1 (deterministic)", TIER1_SCORERS),
        ("Tier 2 (LLM judges)", TIER2_UNIVERSAL | TIER2_TOOL_DEP),
    ]:
        group_results = [
            (k, seen[k])
            for k in sorted(seen)
            if k in group_keys and seen[k].value is not None
        ]
        if not group_results:
            continue
        print(f"\n  {group_name}:")
        for name, a in group_results:
            label = SCORER_LABELS.get(name, name)
            marker = "✓" if _is_pass(a.value) else "✗"
            print(f"    {marker} {label}: {a.value}")
            if not _is_pass(a.value) and a.rationale:
                print(f"      → {a.rationale[:200]}")
    print()

## Interpreting the Results

**Scorers can disagree — and both be right.** On the hotel refusal trace, `AgentGoalAccuracyWithoutReference`(Achieved the user's goal?) returns 0.0 (the user's goal wasn't achieved) while `graceful_refusal`(Correct comply/refuse decision?) returns True (the refusal was correct). They measure different things: one asks "did the user get what they wanted?", the other asks "did the agent make the right decision?" Cross-referencing scorers like this reveals the full picture.

**LLM judges are non-deterministic.** `verification_check` may flag a missing verification in one run and miss it in the next. The agent itself is also non-deterministic — it may call tools differently each run. Run the notebook multiple times to see how stable the results are.

**The failures are intentional.** The system prompt has no instructions to refuse out-of-scope requests or verify saved plans. These gaps create realistic failures for the scorers to catch. In production, you'd fix the prompt and re-evaluate to confirm the fix worked.

**Model choice matters.** These results were produced using `gpt-4.1` as the judge model. Different models may score differently — a weaker model might miss the verification gap or misjudge the refusal. If you swap `JUDGE_MODEL` in the configuration cell, re-run and compare to see how sensitive your scorers are to the judge model.

### What to do next

1. **Inspect traces and evaluation runs in the MLflow UI** at http://localhost:5000. The evaluation results are organized into two parent runs, each with nested child runs:

   | Parent run | Child run | What it contains |
   |---|---|---|
   | `tier1-deterministic-checks` | `pii-scan` | PII detection results (all traces) |
   | | `tool-and-loop-checks` | Tool existence + loop detection (all traces) |
   | | `tool-call-correctness-exact-match` | Exact-match tool correctness (traces with expectations only) |
   | `tier2-llm-judges` | `universal-scorers` | Refusal, grounding, verification, goal accuracy, loop detection (all traces) |
   | | `tool-dependent-scorers` | Tool efficiency + LLM tool correctness (non-refusal traces only) |

   Click into a child run to see its scorer columns — use the column picker to toggle additional scorers if they're not visible by default. You can also click into individual traces in the **Traces** tab to see all assessments attached to that trace across both tiers.

   **Not every trace has the same number of assessments.** Although there are 9 failure modes, a single trace may show anywhere from 8 to 11 assessments depending on its characteristics:

   - **Traces with expectations + tool calls** (e.g., "California parks") get the most — 11 assessments covering all 9 failure modes. Some FMs have both a Tier 1 and Tier 2 scorer (`repeated_action_loop` + `semantic_loop_check`, `tool_call_correctness` + `tool_call_correctness_llm`), which is why the count exceeds 9.
   - **Traces with tool calls but no expectations** (e.g., "3-day Yellowstone trip") get 10 — same as above minus `tool_call_correctness` (exact match), which needs expectations to compare against.
   - **Refusal traces** (e.g., "Book me a hotel") get 8 — they skip `tool_call_correctness` (no expectations), `ToolCallEfficiency`, and `ToolCallCorrectness(LLM)` (tool-dependent scorers don't apply when the agent correctly called zero tools). This means Tool Misuse and Excessive Steps are not assessed on refusal traces, which is correct — you can't misuse tools you didn't call.

2. Iterate on the agent's system prompt to fix the failures the scorers found
3. Re-run this notebook to see if the fixes worked
4. See the [failure mode notebooks](../README.md#failure-modes) for deep dives on each scorer

### Production tip

In production, consider running `graceful_refusal` before the other Tier 2 scorers. If the agent made the wrong comply/refuse decision on a trace, that's a fundamental failure — the other scorers are evaluating behavior that shouldn't have happened. Skipping them saves cost. This notebook keeps all universal scorers in one call for simplicity, but splitting `graceful_refusal` into its own gating step is a worthwhile optimization at scale.